# Notes
- The test does not rely on GPU at all
- This is only a proof-of-concept lab.
- More optimized version will be written with Rust

# Object detection conversion

In [ ]:
common_nouns = [
    # People and Relationships
    "person", "man", "woman", "child", "baby", "boy", "girl", "friend", "neighbor", "guest",
    "teacher", "student", "doctor", "nurse", "patient", "worker", "farmer", "lawyer", "driver", "actor",
    "parent", "father", "mother", "brother", "sister", "son", "daughter", "family", "partner", "boss",
    
    # Home and Furniture
    "house", "home", "room", "kitchen", "bedroom", "bathroom", "livingroom", "garage", "yard", "garden",
    "bed", "pillow", "blanket", "sheet", "closet", "drawer", "wardrobe", "mirror", "desk", "table", 
    "chair", "sofa", "couch", "bench", "stool", "shelf", "bookcase", "door", "window", "wall", 
    "floor", "ceiling", "roof", "gate", "fence", "carpet", "rug", "curtain", "lamp", "bulb",

    # Kitchen, Dining and Food
    "plate", "bowl", "cup", "mug", "glass", "bottle", "jar", "spoon", "fork", "knife", 
    "pot", "pan", "lid", "kettle", "toaster", "blender", "oven", "stove", "microwave", "fridge", 
    "sink", "faucet", "counter", "pantry", "basket", "bin", "trashcan", "sponge", "towel", "napkin",
    "water", "milk", "juice", "coffee", "tea", "bread", "butter", "cheese", "egg", "meat", 
    "chicken", "fish", "rice", "pasta", "flour", "sugar", "salt", "pepper", "soup", "salad",

    # Clothing and Accessories
    "shirt", "tshirt", "sweater", "jacket", "coat", "hoodie", "suit", "pants", "jeans", "shorts", 
    "skirt", "dress", "pajamas", "socks", "shoe", "boot", "sneaker", "sandal", "slipper", "hat", 
    "cap", "glove", "belt", "tie", "bag", "backpack", "purse", "wallet", "suitcase", "umbrella",

    # Technology and Office
    "phone", "smartphone", "charger", "cable", "headphone", "speaker", "computer", "laptop", "monitor", "screen", 
    "keyboard", "mouse", "printer", "tablet", "camera", "lens", "battery", "router", "television", "tv", 
    "remote", "clock", "watch", "calculator", "alarm", "book", "notebook", "pen", "pencil", "marker", 
    "eraser", "ruler", "paper", "sheet", "envelope", "letter", "card", "stamp", "tape", "glue",

    # Vehicles and Places
    "car", "truck", "van", "bus", "taxi", "bicycle", "bike", "motorcycle", "scooter", "train", 
    "airplane", "plane", "boat", "ship", "road", "street", "highway", "sidewalk", "bridge", "station", 
    "airport", "ticket", "passport", "luggage", "map", "building", "office", "factory", "school", "college", 
    "hospital", "clinic", "pharmacy", "store", "shop", "market", "supermarket", "mall", "bank", "hotel", 
    "restaurant", "cafe", "theater", "cinema", "museum", "park", "playground", "church", "farm", "barn"
]

In [27]:
from ultralytics import YOLOE

model = YOLOE("yoloe-26n-seg.pt")
model.set_classes(common_nouns)
model.save_prompt_embeddings("common_nouns.npz")

model = YOLOE("yoloe-26n-seg.pt")
model.load_prompt_embeddings("common_nouns.npz")
model.export(
    format="onnx",
    simplify=True,
    opset=17,
    dynamic=False,
    half=True,

    # Output-specific params
    imgsz=320,
    max_det=50,
    verbose=False,
)

WARNING 'half' is deprecated and will be removed in the future. Use 'quantize' instead.
Ultralytics 8.4.172  Python-3.14.0 torch-2.12.1+cu130 CPU (AMD Ryzen 9 8945HX with Radeon Graphics)
YOLOe-26n-seg summary (fused): 161 layers, 5,103,450 parameters, 0 gradients, 3.0 GFLOPs

PyTorch: starting from 'yoloe-26n-seg.pt' with input shape (1, 3, 320, 320) BCHW and output shape(s) ((1, 276, 2100), (1, 32, 80, 80)) (11.2 MB)

ONNX: starting export with onnx 1.20.0 opset 17...
ONNX: slimming with onnxslim 0.1.82...
ONNX: converting to FP16...
ONNX: export success  0.9s, saved as 'yoloe-26n-seg.onnx' (5.4 MB)

Export complete (1.0s)
Results saved to C:\Users\Bui Thien Nghia\Documents\PERSONAL FILES 2\VGU_CS\Y2\Intro CS\Opsis Gallery\yoloe-26n-seg.onnx
Predict:         yolo predict task=segment model=yoloe-26n-seg.onnx imgsz=320 quantize=16
Validate:        yolo val task=segment model=yoloe-26n-seg.onnx imgsz=320  quantize=16 
Visualize:       https://netron.app


'yoloe-26n-seg.onnx'

In [29]:
import time
import numpy as np
from tqdm import tqdm
from PIL import Image
from ultralytics import YOLOE

model = YOLOE("yoloe-26n-seg.onnx")

def resize(img_path):
    with Image.open(img_path) as img:
        max_size = (720, 720)
        img.thumbnail(max_size, Image.Resampling.LANCZOS)
    return img

inference_time_det = []
for img in tqdm(images):
    start = time.perf_counter()
    res = model.predict(
        img,
        imgsz=320,
        conf=0.5,
        max_det=50,
        verbose=False,
        device="cpu"
    )
    inference_time_det.append(time.perf_counter() - start)

inference_time_det = np.array(inference_time_det) * 1000
print(f"Inference Time in ms:")
print(f"""
Min: {np.min(inference_time_det):.2f} ms
Max: {np.max(inference_time_det):.2f} ms
Avg: {np.mean(inference_time_det):.2f} ms
Med: {np.median(inference_time_det):.2f} ms
Std: {np.std(inference_time_det):.2f} ms
""")
res[0].show()

  0%|          | 0/1000 [00:00<?, ?it/s]

Loading yoloe-26n-seg.onnx for ONNX Runtime inference...
Using ONNX Runtime 1.25.0.dev20260128001 with CPUExecutionProvider


100%|██████████| 1000/1000 [02:27<00:00,  6.77it/s]


Inference Time in ms:

Min: 83.14 ms
Max: 464.09 ms
Avg: 146.51 ms
Med: 141.10 ms
Std: 29.25 ms



# ONNX Optimization

In [14]:
from glob import glob

images = glob("images/*.jpg")
len(images)

1000

In [ ]:
import json
import ast
import numpy as np
import onnxruntime as ort
import cv2

class UniversalYOLOE:
    def __init__(self, model_path: str = "yoloe-26n-seg.onnx"):
        providers = [
            'CoreMLExecutionProvider',
            'CPUExecutionProvider'
        ]
        opts = ort.SessionOptions()
        opts.intra_op_num_threads = 1
        opts.execution_mode = ort.ExecutionMode.ORT_SEQUENTIAL
        opts.graph_optimization_level = ort.GraphOptimizationLevel.ORT_ENABLE_ALL

        self.session = ort.InferenceSession(model_path, opts, providers=providers)
        self.input_name = self.session.get_inputs()[0].name

        meta = self.session.get_modelmeta().custom_metadata_map
        if "names" in meta:
            try:
                class_names = json.loads(meta["names"])
            except json.JSONDecodeError:
                class_names = ast.literal_eval(meta["names"])
        else:
            class_names = {}

        self.class_names = {int(k): v for k, v in class_names.items()}
        self.num_classes = len(class_names)


    def preprocess(self, img_path: str, input_size: int = 320) -> np.ndarray:
        img = cv2.imread(img_path)
        if img is None:
            raise FileNotFoundError(f"Could not load image at {img_path}")
            
        h, w = img.shape[:2]
        scale = input_size / max(h, w)
        new_w, new_h = int(round(w * scale)), int(round(h * scale))
        if (w, h) != (new_w, new_h):
            resized = cv2.resize(img, (new_w, new_h), interpolation=cv2.INTER_LINEAR)
        else:
            resized = img

        pad_h = input_size - new_h
        pad_w = input_size - new_w
        top, bottom = pad_h // 2, pad_h - (pad_h // 2)
        left, right = pad_w // 2, pad_w - (pad_w // 2)
        
        padded = cv2.copyMakeBorder(
            resized, top, bottom, left, right, 
            borderType=cv2.BORDER_CONSTANT, value=(114, 114, 114)
        )

        blob = padded[:, :, ::-1].transpose(2, 0, 1)
        blob = np.ascontiguousarray(blob, dtype=np.float32) / 255.0
        return np.expand_dims(blob, axis=0)

    def pred(self, img_path: str, conf_thresh: float = 0.25) -> list[str]:
        input_tensor = self.preprocess(img_path)
        outputs = self.session.run(None, {self.input_name: input_tensor})[0]
        predictions = outputs[0]  
        if predictions.shape[0] < predictions.shape[1]:
            predictions = predictions.T
        class_scores = predictions[:, 4 : 4 + self.num_classes]
        max_scores = np.max(class_scores, axis=1)
        class_ids = np.argmax(class_scores, axis=1)
        valid_mask = max_scores > conf_thresh
        valid_ids = class_ids[valid_mask]
        unique_ids = list(set(valid_ids.tolist()))
        return [self.class_names.get(cid, f"class_{cid}") for cid in unique_ids]

In [ ]:
import time
from tqdm import tqdm
detector = UniversalYOLOE("yoloe-26n-seg_int8.onnx")

inference_time_det = []
for img in tqdm(images, desc="Testing preprocessing runtime"):
    # try:
        start = time.perf_counter()
        detected_classes = detector.pred(img)
        # print(f"{img}: {detected_classes}")
        inference_time_det.append(time.perf_counter() - start)
    # except Exception: pass

inference_time_det = np.array(inference_time_det) * 1000
print(f"Inference Time in ms:")
print(f"""
Min: {np.min(inference_time_det):.2f} ms
Max: {np.max(inference_time_det):.2f} ms
Avg: {np.mean(inference_time_det):.2f} ms
Med: {np.median(inference_time_det):.2f} ms
Std: {np.std(inference_time_det):.2f} ms
""")